In [ ]:
#@title Setup
import datetime
import glob
import json
import pandas as pd
import os
from IPython.display import Image
from ysngs import Config, WorkFlow, execCmd, curlDownload
yscfg = Config()
yscfg.setEnvs()
from ysngs.prepare import preparInput

In [ ]:
#@title Select WDL script
prefix = 'dlfq' #@param {"type": "string"}
wdl = f'{prefix}.wdl'
script = os.path.join(os.environ['HYM_SCRIPT'], 'wdl', wdl)
## Make workflow instance
wf = WorkFlow(script)
wf.check()

In [ ]:
#@title Make input(s)
# Set input
list_file = ''  #@param {type: 'string'}
data_ids = ''  #@param {type: 'string'}
splitted = True #@param {type: 'boolean'}
out_dir = '' #@param {type: 'string'}
thread = 16 #@param {type: 'raw'}
## Prepare input 
input_path = preparInput(prefix, {
    'ids': data_ids.split(',') if data_ids != '' else [],
    'list': list_file,
    'out_dir': os.path.join(os.environ['HYM_DATA'], out_dir),
    'split_file': splitted,
    'thread': thread
})

In [ ]:
#@title Run (Use SRA toolkit)
inputs = json.load(open(input_path))
for input in inputs:
    now = datetime.datetime.now()
    path = os.path.join(os.environ['HYM_TEMP'], f"{now.strftime('%Y-%m-%d_%H-%M-%S')}.json")
    json.dump(input, open(path, 'w'))
    # Run
    wf.run(path)

In [ ]:
#@title Visualize
## Set image file path
graph_path = os.path.join(os.environ['HYM_LOG'], f"{os.path.split(wf.script)[1].replace('.wdl', '.dot')}")
img_path = graph_path.replace('.dot', '.png')
## Make dot file
wf.graph(graph_path, detailed=True)
## Display
display(Image(img_path))